# Additional End of week Exercise - week 2

Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.

### This is a Multi LLM Japanese AI Tutor 
This calls three different LLMs that are able to read, translate, and explain Japanese setences. The models are shown side by side for comparison of each response. They return conversational history for each instance.


In [ ]:
# imports as always
import os
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI
import gradio as gr 
from concurrent.futures import ThreadPoolExecutor

In [ ]:
#API key check are set, just in case and call the Python CLient Library for OpenAI 

load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY') 
ollama_api_key = "ollama"

if openai_api_key:
    print(f"OpenAI API key exists and begins {openai_api_key[:8]}")
else:
    print(f"OpenAI API key not set")

if anthropic_api_key:
    print(f"Anthropic API key exists and begins {anthropic_api_key[:8]}")
else:
    print(f"Anthropic API key not set")

#URL for models 
anthropic_url = "https://api.anthropic.com/v1/"
ollama_url ="http://localhost:11434/v1"

MODEL_GPT = "gpt-4.1-mini"
MODEL_CLAUDE ="claude-sonnet-4-5-20250929"
MODEL_OLLAMA = "llama3.2"

#Set variables using the library
openai=OpenAI()
anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)
ollama = OpenAI(api_key=ollama_api_key, base_url=ollama_url)

In [ ]:
#Defining AI tutor prompts for the system and user 
system_prompt = """
You are a expert language tutor in Japanese and English.
You can read and understand both sentences to explain the nuance of 
the language such the vocabulary, grammar structures, and how kanji
and their radicals are used for explanation. If you don't know an answer,
so say so.
"""
user_prompt = """
I am the student who is still learning Japanese as a second language.
I know English as a native, I am still having trouble understanding 
Japanese grammar, especially as the construction of words turn verbs
into past, present tense, negatives, etc. continue to add complexity and
grow in size.  
"""

In [ ]:
#Each functions runs with the message prompt and retains history 

def gptChat(message, history):
    messages = [{"role":"system", "content": system_prompt}] + history + [{"role": "user", "content":message}]
    response = openai.chat.completions.create(model=MODEL_GPT, messages=messages)
    return response.choices[0].message.content

def ollamaChat(message, history):
    messages = [{"role":"system", "content": system_prompt}] + history + [{"role": "user", "content":message}]
    response = ollama.chat.completions.create(model=MODEL_OLLAMA, messages=messages)
    return response.choices[0].message.content

def claudeChat(message,history):
    messages = [{"role":"system", "content": system_prompt}] + history + [{"role": "user", "content":message}]
    response = anthropic.chat.completions.create(model=MODEL_CLAUDE, messages=messages)
    return response.choices[0].message.content

#run models concurrently instead of one by one, given to gradio to run each function for each model
def compare_Models(message, ollama_hist, gpt_hist, claude_hist):

    with ThreadPoolExecutor() as executor:
        response1 = executor.submit(ollamaChat,message, ollama_hist)
        response2 = executor.submit(gptChat,message, gpt_hist)
        response3 = executor.submit(claudeChat, message, claude_hist)

        ollama_response = response1.result()
        gpt_response = response2.result()
        claude_response =response3.result()

    new_ollama_hist = ollama_hist + [
        {"role": "user", "content": message},
        {"role": "assistant", "content": ollama_response},
    ]
    new_gpt_hist = gpt_hist + [
        {"role": "user", "content": message},
        {"role": "assistant", "content": gpt_response},
    ]
    new_claude_hist = claude_hist + [
        {"role": "user", "content": message},
        {"role": "assistant", "content": claude_response},
    ]
    return (
        ollama_response, gpt_response, claude_response,
        new_ollama_hist, new_gpt_hist, new_claude_hist,
    )

#gradio UI displays three rows of the different LLMS
with gr.Blocks() as ui:
    gr.Markdown("# Multi-LLM Japanese Language Comparison")
    #Saves history of conversations for each model
    ollama_history = gr.State([])
    gpt_history = gr.State([])
    claude_history = gr.State([])

    user_input = gr.Textbox(
            label ="Ask a question",
            placeholder ="Type something..."
        )
    submit_button = gr.Button("Go")

    with gr.Row():
        with gr.Column():
            gr.Markdown(f"### {MODEL_OLLAMA}")
            ollama_output =gr.Markdown(label="### Llama")
        with gr.Column():
            gr.Markdown(f"### {MODEL_GPT}")
            gpt_output = gr.Markdown(label="### GPT")
        with gr.Column():
            gr.Markdown(f"### {MODEL_CLAUDE}")
            claude_output = gr.Markdown(label="### Claude")

    submit_button.click(
        fn =compare_Models,
        inputs=[user_input, ollama_history, gpt_history, claude_history],
        outputs =[
            ollama_output,gpt_output,claude_output,
            ollama_history,gpt_history,claude_history
        ]
    )


In [ ]:
ui.launch(inbrowser=True)